# Labwork 3
This new labwork focuses on the SCAN pattern, which is the Swiss-knife of the little parallel programmer...

The three exercises are dealing with the Inclusive SCAN version, from left to right (no reverse).

Run the following cell first. It reports what hardware you were given and stops the notebook if that is a CPU: the runtime is declared in this file, but should you land on a CPU anyway, go to **Runtime ▸ Change runtime type ▸ T4 GPU** and run it again.

In [13]:
!nvidia-smi
!nvcc --version

import importlib.util
import subprocess
import sys

if importlib.util.find_spec("numba") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numba"],
                   check=True)

import numba

try:
    # Recent numba keeps the CUDA target in a separate distribution; the one
    # Colab ships still has it built in. Ask, rather than assume either way.
    from numba import config, cuda
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numba-cuda"],
                   check=True)
    from numba import config, cuda

config.CUDA_ENABLE_PYNVJITLINK = True

print(f"numba {numba.__version__}")

if not cuda.is_available():
    raise SystemExit(
        "No CUDA device: Runtime -> Change runtime type -> T4 GPU, "
        "then Runtime -> Run all. Nothing below this cell can run on a CPU."
    )
cuda.detect()

/bin/bash: line 1: nvidia-smi: command not found
/bin/bash: line 1: nvcc: command not found
numba 0.67.0
Found 1 CUDA devices
id 0    NVIDIA GeForce RTX 5060 Ti                              [SUPPORTED]
                      Compute Capability: 12.0
                           PCI Device ID: 0
                              PCI Bus ID: 1
                                    UUID: GPU-fd7a79a8-dba9-c363-20c2-85cfb7c05ea9
                                Watchdog: Enabled
             FP32/FP64 Performance Ratio: 64
                            FP16 Support: Yes
                            BF16 Support: Yes
                             FP8 Support: Yes
Summary:
	1/1 devices are supported


True

### Checking your work

The cell below defines two helpers, provided: there is nothing to fill in.

`verify(...)` compares your result with a NumPy reference and **raises** when
they disagree, naming the first index that differs. A red cell means a wrong
kernel, a green one a right one.

`benchmark(...)` times the GPU against NumPy over a range of sizes, after
warm-up runs that pay for the JIT compilation: timing the first launch measures
the compiler, not the kernel.

Each exercise is followed by its own **`testN`**, which runs your class and
hands the result to the oracle for exercise *N*. You may call `test1(...)`
again at any point, on any array you like, and it still tests exercise 1.

In [14]:
# --- provided: the self-check and benchmark helpers --------------------------
import contextlib
import io
from time import perf_counter

import numpy as np
from numba import core, cuda

# The first exercises launch a single block on purpose, and numba warns about
# it every time. The warning is right in general and noise here.
core.config.CUDA_LOW_OCCUPANCY_WARNINGS = False


class CheckFailed(AssertionError):
    """Raised when a kernel disagrees with its CPU reference."""


# Named `verify` rather than `check` because each exercise gets its own
# `checkN`: that one is the oracle for exercise N, this one is the primitive
# they are all written in terms of.


def verify(name, got, expected, tolerance=0.0):
    """Compare a result against its oracle, and raise on the first difference.

    `tolerance` is relative; leave it at 0 for integers, where the answer is
    exact, and pass something like 1e-5 for floats.
    """
    got, expected = np.asarray(got), np.asarray(expected)
    if got.shape != expected.shape:
        raise CheckFailed(
            f"{name}: shape is {got.shape}, expected {expected.shape}"
        )
    if tolerance:
        wrong = ~np.isclose(got, expected, rtol=tolerance, atol=tolerance)
    else:
        wrong = got != expected
    count = int(np.count_nonzero(wrong))
    if count:
        first = int(np.argmax(wrong))
        raise CheckFailed(
            f"{name}: {count} of {got.size} values are wrong. "
            f"First at index {first}: got {got.ravel()[first]}, "
            f"expected {expected.ravel()[first]}."
        )
    print(f"PASS  {name}  ({got.size} value{'' if got.size == 1 else 's'})")


def _time_once(call):
    """One timed call, in milliseconds, with the device drained first.

    Most of these classes print their own launch sizes and timings on every
    call, which would bury the table below under a few hundred lines, so the
    timed call is run with its output swallowed.
    """
    with contextlib.redirect_stdout(io.StringIO()):
        start = perf_counter()
        call()
        cuda.synchronize()
        stop = perf_counter()
    return (stop - start) * 1000.0


def peak_bandwidth():
    """The card's theoretical memory bandwidth in GB/s, or None if unreadable.

    Peak is what a reduction or a scan should be measured against. These
    kernels do almost no arithmetic: they read the array once and produce a
    handful of numbers, so the only question worth asking is how close to the
    memory system they get. A speed-up against one CPU core answers a much
    easier question.
    """
    try:
        device = cuda.get_current_device()
        clock_khz = device.MEMORY_CLOCK_RATE            # kHz
        bus_bytes = device.GLOBAL_MEMORY_BUS_WIDTH / 8  # bits -> bytes
        return 2 * clock_khz * 1e3 * bus_bytes / 1e9    # 2: DDR
    except Exception:
        return None


def benchmark(name, build, sizes, repeats=10, warmup=3):
    """Time the GPU against NumPy over `sizes`.

    `build(size)` returns `(gpu, cpu)`, or `(gpu, cpu, nbytes)` to get the
    bandwidth columns as well. The two callables take no arguments and already
    hold their data, so the upload is not timed unless the kernel needs it.
    `nbytes` is the traffic one call claims to move; what counts as traffic is
    said where each `build` is written, because it is not the same question for
    a reduction, a scan and a sort.
    """
    peak = peak_bandwidth()
    print(name + (f"   —   this card peaks at {peak:.0f} GB/s" if peak else ""))
    print(f"{'size':>12} | {'GPU (ms)':>9} | {'NumPy (ms)':>11} | "
          f"{'speed-up':>9} | {'GB/s':>7} | {'of peak':>8}")
    print("-" * 72)
    for size in sizes:
        with contextlib.redirect_stdout(io.StringIO()):
            built = build(size)
            gpu, cpu = built[0], built[1]
            nbytes = built[2] if len(built) > 2 else None
            for _ in range(warmup):
                gpu()
            cuda.synchronize()
        best = min(_time_once(gpu) for _ in range(repeats))
        reference = min(_time_once(cpu) for _ in range(3))

        rate = None if nbytes is None else nbytes / (best / 1000.0) / 1e9
        rate_text = "      -" if rate is None else f"{rate:7.1f}"
        share_text = ("       -" if rate is None or peak is None
                      else f"{100 * rate / peak:7.1f}%")
        print(f"{size:12d} | {best:9.3f} | {reference:11.3f} | "
              f"{reference / best:8.1f}x | {rate_text} | {share_text}")

## Exercise 1
In this exercise, you must implement a **warp-scan**. 
It is a scan into a single warp for a single block, and so for a small amount of data (32 values). 
Then, you should implement the PRAM algorithm, without any need for synchronisation...

NB: you must write the three device functions:
- `load_shared_memory`
- `pointer_jumping`
- `save_shared_memory`
The kernel should be read, too...

In [15]:
from __future__ import annotations

import numpy as np
from numba.np.numpy_support import from_dtype
from numba import cuda, core


class WarpScan(object):
    """Create a scan object that scans values using a given binary
    function. The binary function is compiled once and cached inside this
    object. Keeping this object alive will prevent re-compilation.

    The scan is functional for a limited number of values, as it runs into a single warp.
    The number of values should be less or equal than 32.
    """

    _cache = {}

    _WARP_SIZE = 32
    _NUM_WARPS = 1

    @staticmethod
    def _gpu_kernel_factory(fn, np_type):
        """Factory of kernels for the block-scan problem...

        This function returns a Cuda Kernel that does block-scan of some data using a given binary functor."""

        scan_op = cuda.jit(device=True)(fn)

        max_block_size = WarpScan._WARP_SIZE

        @cuda.jit(device=True)
        def load_shared_memory(shared_memory, d_input):
            local_tid = cuda.threadIdx.x

            # Load data into shared memory.
            if local_tid < d_input.size:
                shared_memory[local_tid] = d_input[local_tid]

        @cuda.jit(device=True)
        def pointer_jumping(shared_memory, jump):
            tid = cuda.threadIdx.x

            # Inclusive scan using pointer jumping.
            if tid >= jump:
                shared_memory[tid] = scan_op(
                    shared_memory[tid - jump],
                    shared_memory[tid]
                )

        @cuda.jit(device=True)
        def save_shared_memory(shared_memory, d_output):
            local_tid = cuda.threadIdx.x

            # Save data from shared memory.
            if local_tid < d_output.size:
                d_output[local_tid] = shared_memory[local_tid]

        def gpu_scan_block(d_input, d_output):
            """
            Per block SCAN...
            """
            shared_memory = cuda.shared.array(
                shape=max_block_size,
                dtype=np_type
            )

            # Load input into shared memory.
            load_shared_memory(shared_memory, d_input)

            # PRAM pointer jumping.
            jump = 1
            while jump < d_input.size:
                pointer_jumping(shared_memory, jump)
                cuda.syncwarp()
                jump *= 2

            # Save the scan result.
            save_shared_memory(shared_memory, d_output)

        return cuda.jit(gpu_scan_block)

    def __init__(self, functor):
        """
        :param functor: A function implementing a binary operation for
                        scan. It will be compiled as a CUDA device
                        function using ``cuda.jit(device=True)``.
        """
        self._functor = functor

    def _compile(self, dtype):
        key = self._functor, dtype
        if key not in self._cache:
            self._cache[key] = WarpScan._gpu_kernel_factory(
                self._functor, from_dtype(dtype)
            )
        return self._cache[key]

    def __call__(
        self,
        d_input,
        d_output,
        stream=cuda.default_stream()
    ):
        """ Performs a per-block SCAN.

        :param d_input: A host or device array.
        :param stream: Optional CUDA stream in which to perform the scan.
                       If no stream is specified, the default stream of 0 is used.
        """

        # ensure 1d array
        if d_input.ndim != 1:
            raise TypeError("only support 1D array")

        # ensure size > 0
        if d_input.size < 1:
            raise ValueError("array's length is 0")

        # ensure arrays' size are the same
        if d_input.size != d_output.size:
            raise ValueError(
                f"arrays' length are different ({d_input.size} / {d_output.size})"
            )

        # ensure size <= 32
        max_block_size = WarpScan._WARP_SIZE
        if d_input.size > max_block_size:
            raise ValueError(
                "array's length is greater than max block size"
            )

        _sav, core.config.CUDA_LOW_OCCUPANCY_WARNINGS = (
            core.config.CUDA_LOW_OCCUPANCY_WARNINGS,
            False
        )

        kernel = self._compile(d_input.dtype)

        # Perform the scan on the GPU
        start_event = cuda.event(True)
        start_event.record(stream=stream)

        nb_threads = max_block_size
        nb_blocks = 1
        kernel[nb_blocks, nb_threads, stream](d_input, d_output)

        stop_event = cuda.event(True)
        stop_event.record(stream=stream)
        stop_event.synchronize()
        ct = cuda.event_elapsed_time(start_event, stop_event)

        core.config.CUDA_LOW_OCCUPANCY_WARNINGS = _sav

        return d_output.copy_to_host(), ct


#### Tests — exercise 1 — the scan of one warp

In [16]:
def test1(h_input, scan_class=WarpScan):
    def check1(result):
        """A scan is right when every prefix is.

        Random values, not ones: with ones, a scan that sums the wrong
        neighbours still prints 1, 2, 3, ... and looks perfect.
        """
        verify(f"inclusive scan of {h_input.size} values", result,
               np.cumsum(h_input))

    scanner = scan_class(lambda a, b: a + b)
    d_output = cuda.device_array(shape=h_input.size, dtype=h_input.dtype)
    result, ct = scanner(cuda.to_device(h_input), d_output)
    check1(result)
    return ct


rng = np.random.default_rng(1)
test1(rng.integers(0, 1 << 10, size=32, dtype=np.int32))

PASS  inclusive scan of 32 values  (32 values)


110.54057312011719

## Exercise 2
In this exercise, you must implement a **block-scan**. 
It is a scan into a single block, for a small amount of data (256 values). 
Then, you should implement the PRAM algorithm, with the correct per-warp synchronizations...

NB: you must write the three device functions:
- `load_shared_memory`
- `pointer_jumping`
- `save_shared_memory`
The kernel should be read, too...

In [17]:
from __future__ import annotations

import numpy as np
from numba.np.numpy_support import from_dtype
from numba import cuda, core


class BlockScan(object):
    """Create a scan object that scans values using a given binary
    function. The binary function is compiled once and cached inside this
    object. Keeping this object alive will prevent re-compilation.

    The scan is functional for a limited number of values, as it runs into a single block.
    The number of values should be less than 256.
    """

    _cache = {}

    _WARP_SIZE = 32
    _NUM_WARPS = 8

    @staticmethod
    def _gpu_kernel_factory(fn, np_type):
        """Factory of kernels for the block-scan problem...

        This function returns a Cuda Kernel that does block-scan of some data using a given binary functor."""

        scan_op = cuda.jit(device=True)(fn)

        max_block_size = BlockScan._NUM_WARPS * BlockScan._WARP_SIZE

        @cuda.jit(device=True)
        def load_shared_memory(shared_memory, d_input):
            local_tid = cuda.threadIdx.x

            # Load data into shared memory.
            if local_tid < d_input.size:
                shared_memory[local_tid] = d_input[local_tid]

        @cuda.jit(device=True)
        def pointer_jumping(shared_memory, jump):
            tid = cuda.threadIdx.x

            # Read the previous value before updating shared memory.
            if tid >= jump:
                temp = shared_memory[tid - jump]
            else:
                temp = shared_memory[tid]

            cuda.syncthreads()

            if tid >= jump:
                shared_memory[tid] = scan_op(
                    temp, shared_memory[tid]
                )

        @cuda.jit(device=True)
        def save_shared_memory(shared_memory, d_output):
            local_tid = cuda.threadIdx.x

            # Save data from shared memory.
            if local_tid < d_output.size:
                d_output[local_tid] = shared_memory[local_tid]

        def gpu_scan_block(d_input, d_output):
            """
            Per block SCAN...
            """
            # Move data to shared memory.
            shared_memory = cuda.shared.array(
                shape=max_block_size,
                dtype=np_type
            )
            load_shared_memory(shared_memory, d_input)

            # Initialize unused positions.
            tid = cuda.threadIdx.x
            if tid >= d_input.size:
                shared_memory[tid] = 0

            cuda.syncthreads()

            # Pointer jumping.
            jump = 1
            while jump < d_input.size:
                pointer_jumping(shared_memory, jump)
                cuda.syncthreads()
                jump *= 2

            # Now store the result.
            save_shared_memory(shared_memory, d_output)

        return cuda.jit(gpu_scan_block)

    def __init__(self, functor):
        """
        :param functor: A function implementing a binary operation for
                        scan. It will be compiled as a CUDA device
                        function using ``cuda.jit(device=True)``.
        """
        self._functor = functor

    def _compile(self, dtype):
        key = self._functor, dtype
        if key not in self._cache:
            self._cache[key] = BlockScan._gpu_kernel_factory(
                self._functor, from_dtype(dtype)
            )
        return self._cache[key]

    def __call__(
        self,
        d_input,
        d_output,
        stream=cuda.default_stream()
    ):
        """ Performs a per-block SCAN.

        :param d_input: A host or device array.
        :param stream: Optional CUDA stream in which to perform the scan.
                       If no stream is specified, the default stream of 0 is used.
        """

        # ensure 1d array
        if d_input.ndim != 1:
            raise TypeError("only support 1D array")

        # ensure size > 0
        if d_input.size < 1:
            raise ValueError("array's length is 0")

        # ensure arrays' size are the same
        if d_input.size != d_output.size:
            raise ValueError(
                f"arrays' length are different ({d_input.size} / {d_output.size})"
            )

        # ensure size <= 256
        max_block_size = BlockScan._WARP_SIZE * BlockScan._NUM_WARPS
        if d_input.size > max_block_size:
            raise ValueError("array's length is greater than max block size")

        _sav, core.config.CUDA_LOW_OCCUPANCY_WARNINGS = (
            core.config.CUDA_LOW_OCCUPANCY_WARNINGS,
            False
        )

        kernel = self._compile(d_input.dtype)

        # Perform the scan on the GPU.
        start_event = cuda.event(True)
        start_event.record(stream=stream)

        nb_threads = max_block_size
        nb_blocks = 1
        kernel[nb_blocks, nb_threads, stream](d_input, d_output)

        stop_event = cuda.event(True)
        stop_event.record(stream=stream)
        stop_event.synchronize()
        ct = cuda.event_elapsed_time(start_event, stop_event)

        core.config.CUDA_LOW_OCCUPANCY_WARNINGS = _sav

        return d_output.copy_to_host(), ct

#### Tests — exercise 2 — the scan of one block

In [18]:
def test2(h_input, scan_class=BlockScan):
    def check2(result):
        verify(f"inclusive scan of {h_input.size} values", result,
               np.cumsum(h_input))

    scanner = scan_class(lambda a, b: a + b)
    d_output = cuda.device_array(shape=h_input.size, dtype=h_input.dtype)
    result, ct = scanner(cuda.to_device(h_input), d_output)
    check2(result)
    return ct


rng = np.random.default_rng(2)
for size in (32, 100, 256):          # 100 is not a multiple of the warp size
    test2(rng.integers(0, 1 << 10, size=size, dtype=np.int32))

PASS  inclusive scan of 32 values  (32 values)
PASS  inclusive scan of 100 values  (100 values)
PASS  inclusive scan of 256 values  (256 values)


## Exercise 3
This exercise deals with a block SCAN, but here applying the **hybrid** strategy saw in lecture 3. 

The idea is then (for a single block):
- to load the data in shared memory
- to do a SCAN per warp (so without warp synchronization)
- to do a warp synchronization
- to do a SCAN considering the last value of each warp (into a single warp, so no warp synchronization)
- to do a warp synchronization
- to apply the final MAP for all warps except the first...

Try this wonderful strategy below.

NB: the computation time will be roughly speaking the same, but the idea is to apply this algorithm in simple condition first, before to do it at a larger scale.

In [19]:
from __future__ import annotations

import numpy as np
from numba.np.numpy_support import from_dtype
from numba import cuda, core


class BlockScanHybrid(object):
    """Create a scan object that scans values using a given binary
    function. The binary function is compiled once and cached inside this
    object. Keeping this object alive will prevent re-compilation.

    The scan is functional for a limited number of values, as it runs into a single block.
    The number of values should be less than 256.
    """

    _cache = {}

    _WARP_SIZE = 32
    _NUM_WARPS = 8

    @staticmethod
    def _gpu_kernel_factory(fn, np_type):
        """Factory of kernels for the block-scan problem."""

        scan_op = cuda.jit(device=True)(fn)

        max_block_size = (
            BlockScanHybrid._NUM_WARPS
            * BlockScanHybrid._WARP_SIZE
        )

        @cuda.jit(device=True)
        def load_shared_memory(shared_memory, d_input):
            local_tid = cuda.threadIdx.x

            if local_tid < d_input.size:
                shared_memory[local_tid] = d_input[local_tid]

        @cuda.jit(device=True)
        def scan_per_warp(shared_memory):
            tid = cuda.threadIdx.x
            warp_tid = tid & 31

            jump = 1
            while jump < 32:
                if warp_tid >= jump:
                    temp = shared_memory[tid - jump]
                else:
                    temp = shared_memory[tid]

                cuda.syncwarp()

                if warp_tid >= jump:
                    shared_memory[tid] = scan_op(
                        temp, shared_memory[tid]
                    )

                cuda.syncwarp()
                jump *= 2

        @cuda.jit(device=True)
        def get_extra_value(shared_memory, last):
            tid = cuda.threadIdx.x
            warp_tid = tid & 31
            warp_id = tid >> 5

            if warp_tid == 31:
                last[warp_id] = shared_memory[tid]

        @cuda.jit(device=True)
        def apply_final_map(shared_memory, last):
            tid = cuda.threadIdx.x
            warp_id = tid >> 5

            if warp_id > 0:
                shared_memory[tid] = scan_op(
                    last[warp_id - 1],
                    shared_memory[tid]
                )

        @cuda.jit(device=True)
        def save_shared_memory(shared_memory, d_output):
            local_tid = cuda.threadIdx.x

            if local_tid < d_output.size:
                d_output[local_tid] = shared_memory[local_tid]

        def gpu_scan_block(d_input, d_output):
            """
            Per block SCAN...
            """
            # Move data to shared memory.
            shared_memory = cuda.shared.array(
                shape=max_block_size,
                dtype=np_type
            )
            extra_shared_memory = cuda.shared.array(
                shape=32,
                dtype=np_type
            )

            load_shared_memory(shared_memory, d_input)

            tid = cuda.threadIdx.x

            # Initialize unused input positions.
            if tid >= d_input.size:
                shared_memory[tid] = 0

            cuda.syncthreads()

            # Scan each warp independently.
            scan_per_warp(shared_memory)
            cuda.syncthreads()

            # Collect the last value of each warp.
            get_extra_value(shared_memory, extra_shared_memory)
            cuda.syncthreads()

            # Scan the warp totals using the first warp.
            if tid < 32:
                jump = 1
                while jump < 8:
                    if tid < 8 and tid >= jump:
                        temp = extra_shared_memory[tid - jump]
                    else:
                        temp = extra_shared_memory[tid]

                    cuda.syncwarp()

                    if tid < 8 and tid >= jump:
                        extra_shared_memory[tid] = scan_op(
                            temp, extra_shared_memory[tid]
                        )

                    cuda.syncwarp()
                    jump *= 2

            cuda.syncthreads()

            # Apply the prefix from previous warps.
            apply_final_map(shared_memory, extra_shared_memory)
            cuda.syncthreads()

            # Save the result.
            save_shared_memory(shared_memory, d_output)

        return cuda.jit(gpu_scan_block)

    def __init__(self, functor):
        """
        :param functor: A function implementing a binary operation for
                        scan. It will be compiled as a CUDA device
                        function using ``cuda.jit(device=True)``.
        """
        self._functor = functor

    def _compile(self, dtype):
        key = self._functor, dtype
        if key not in self._cache:
            self._cache[key] = BlockScanHybrid._gpu_kernel_factory(
                self._functor, from_dtype(dtype)
            )
        return self._cache[key]

    def __call__(
        self,
        d_input,
        d_output=None,
        stream=cuda.default_stream()
    ):
        """ Performs a per-block SCAN.

        :param d_input: A host or device array.
        :param stream: Optional CUDA stream in which to perform the scan.
                       If no stream is specified, the default stream of 0 is used.
        """

        # Ensure 1D array.
        if d_input.ndim != 1:
            raise TypeError("only support 1D array")

        # Ensure size > 0.
        if d_input.size < 1:
            raise ValueError("array's length is 0")

        # Allocate output if not provided.
        if d_output is None:
            d_output = cuda.device_array(
                shape=d_input.size,
                dtype=d_input.dtype,
                stream=stream
            )

        # Ensure arrays' size are the same.
        if d_input.size != d_output.size:
            raise ValueError(
                f"arrays' length are different "
                f"({d_input.size} / {d_output.size})"
            )

        # Ensure size <= 256.
        max_block_size = (
            BlockScanHybrid._WARP_SIZE
            * BlockScanHybrid._NUM_WARPS
        )
        if d_input.size > max_block_size:
            raise ValueError(
                "array's length is greater than max block size"
            )

        _sav, core.config.CUDA_LOW_OCCUPANCY_WARNINGS = (
            core.config.CUDA_LOW_OCCUPANCY_WARNINGS,
            False
        )

        kernel = self._compile(d_input.dtype)

        # Perform the scan on the GPU.
        start_event = cuda.event(True)
        start_event.record(stream=stream)

        nb_threads = max_block_size
        nb_blocks = 1
        kernel[nb_blocks, nb_threads, stream](
            d_input, d_output
        )

        stop_event = cuda.event(True)
        stop_event.record(stream=stream)
        stop_event.synchronize()
        ct = cuda.event_elapsed_time(start_event, stop_event)

        core.config.CUDA_LOW_OCCUPANCY_WARNINGS = _sav

        return d_output.copy_to_host(stream=stream), ct

#### Tests — exercise 3 — the hybrid block scan

In [20]:
def test3(h_input, scan_class=BlockScanHybrid, tolerance=0.0):
    def check3(result):
        verify(f"hybrid scan of {h_input.size} values", result,
               np.cumsum(h_input), tolerance=tolerance)

    scanner = scan_class(lambda a, b: a + b)
    d_output = cuda.device_array(shape=h_input.size, dtype=h_input.dtype)
    result, ct = scanner(cuda.to_device(h_input), d_output)
    check3(result)
    return ct


rng = np.random.default_rng(3)
for size in (32, 100, 256):
    test3(rng.integers(0, 1 << 10, size=size, dtype=np.int32))

# Float32: addition is not associative there, so the two orders differ in the
# last bits and the comparison needs a tolerance.
test3(rng.random(256).astype(np.float32), tolerance=1e-5)

PASS  hybrid scan of 32 values  (32 values)
PASS  hybrid scan of 100 values  (100 values)
PASS  hybrid scan of 256 values  (256 values)
PASS  hybrid scan of 256 values  (256 values)


294.8033752441406

## Exercise 4
In this exercise you must implement the **full scan** for a given binary associative operator.

The lecture 3 gives you all the clue for this purpose:
- the per block SCAN, with its extra value,
- the inter-block extra value scan which is a recursive call,
- the MAP to end the SCAN by adding the extra block scanned values (with shift). 

In [21]:
from __future__ import annotations

import numpy as np
from numba.np.numpy_support import from_dtype
from numba import cuda, core


def display(array, all_values=False):
    def aprint(data_range, noend=False):
        for i in data_range:
            print(f"{array[i]} ", end="")
            if (i % 16) == 15:
                print("")
        if not noend:
            print("...")

    if all_values:
        aprint(range(array.size), noend=True)
        return
    aprint(range(0, min(array.size, 16)))
    if array.size < 256 - 16:
        return
    aprint(range(256 - 16, min(array.size, 256 + 16)))
    if array.size < 512 - 16:
        return
    aprint(range(512 - 16, min(array.size, 512 + 16)))
    if array.size < 768 - 16:
        return
    aprint(range(768 - 16, min(array.size, 768 + 16)))
    aprint(range(array.size - 16 - 256, array.size - 256 + 16))
    aprint(range(array.size - 16, array.size), noend=True)


class InclusiveScan(object):
    """Create a scan object that scans values using a given binary
    function. The binary function is compiled once and cached inside this
    object. Keeping this object alive will prevent re-compilation.

    The scan is functional for a limited number of values, as it runs into a single block.
    The number of values should be less than 256.
    """

    _kernels_block_scan = {}
    _kernels_block_map = {}

    _WARP_SIZE = 32
    _NUM_WARPS = 8

    @staticmethod
    def _gpu_kernel_block_scan_factory(fn, np_type):
        """Factory of kernels for the block-scan problem."""

        scan_op = cuda.jit(device=True)(fn)

        max_block_size = InclusiveScan._NUM_WARPS * InclusiveScan._WARP_SIZE

        @cuda.jit(device=True)
        def load_shared_memory(shared_memory, d_input):
            local_tid = cuda.threadIdx.x
            tid = cuda.grid(1)

            if tid < d_input.size:
                shared_memory[local_tid] = d_input[tid]

        @cuda.jit(device=True)
        def pointer_jumping(shared_memory, jump):
            tid = cuda.threadIdx.x

            if tid >= jump:
                temp = shared_memory[tid - jump]
            else:
                temp = shared_memory[tid]

            cuda.syncthreads()

            if tid >= jump:
                shared_memory[tid] = scan_op(
                    temp, shared_memory[tid]
                )

            cuda.syncthreads()

        @cuda.jit(device=True)
        def save_shared_memory(shared_memory, d_output, d_extras):
            local_tid = cuda.threadIdx.x
            global_tid = cuda.grid(1)

            if global_tid < d_output.size:
                d_output[global_tid] = shared_memory[local_tid]

            # Save the final valid prefix of this block.
            if (
                local_tid == cuda.blockDim.x - 1
                or global_tid == d_output.size - 1
            ):
                d_extras[cuda.blockIdx.x] = shared_memory[local_tid]

        def gpu_scan_block(d_input, d_output, d_extras):
            """
            Per block SCAN...
            """
            # Move data to shared memory.
            shared_memory = cuda.shared.array(
                shape=max_block_size, dtype=np_type
            )
            load_shared_memory(shared_memory, d_input)
            cuda.syncthreads()

            # Pointer jumping across the whole block.
            jump = 1
            while jump < cuda.blockDim.x:
                pointer_jumping(shared_memory, jump)
                jump *= 2

            # Store the result.
            save_shared_memory(shared_memory, d_output, d_extras)

        return cuda.jit(gpu_scan_block)

    def __init__(self, functor):
        """
        :param functor: A function implementing a binary operation for
                        scan. It will be compiled as a CUDA device
                        function using ``cuda.jit(device=True)``.
        """
        self._functor = functor

    def _compile_block_scan(self, dtype):
        key = self._functor, dtype
        if key not in self._kernels_block_scan:
            self._kernels_block_scan[key] = (
                InclusiveScan._gpu_kernel_block_scan_factory(
                    self._functor, from_dtype(dtype)
                )
            )
        return self._kernels_block_scan[key]

    @staticmethod
    def _gpu_kernel_block_map_factory(fn, np_type):
        """Factory of kernels for the block-map problem."""

        scan_op = cuda.jit(device=True)(fn)

        def gpu_map_block(d_io, d_scan):
            """
            Per block MAP...
            """
            global_tid = cuda.grid(1)

            if global_tid < d_io.size and cuda.blockIdx.x > 0:
                d_io[global_tid] = scan_op(
                    d_scan[cuda.blockIdx.x - 1],
                    d_io[global_tid]
                )

        return cuda.jit(gpu_map_block)

    def _compile_block_map(self, dtype):
        key = self._functor, dtype
        if key not in self._kernels_block_map:
            self._kernels_block_map[key] = (
                InclusiveScan._gpu_kernel_block_map_factory(
                    self._functor, from_dtype(dtype)
                )
            )
        return self._kernels_block_map[key]

    def __call__(self, d_input, d_output, stream=cuda.default_stream()):
        """Performs a full inclusive SCAN."""

        # Ensure 1D array.
        if d_input.ndim != 1:
            raise TypeError("only support 1D array")

        # Ensure size > 0.
        if d_input.size < 1:
            raise ValueError("array's length is 0")

        # Ensure arrays' size are the same.
        if d_input.size != d_output.size:
            raise ValueError(
                f"arrays' length are different "
                f"({d_input.size} / {d_output.size})"
            )

        _sav, core.config.CUDA_LOW_OCCUPANCY_WARNINGS = (
            core.config.CUDA_LOW_OCCUPANCY_WARNINGS,
            False
        )

        kernel_step1 = self._compile_block_scan(d_input.dtype)
        kernel_step2 = self._compile_block_map(d_input.dtype)

        max_block_size = InclusiveScan._WARP_SIZE * InclusiveScan._NUM_WARPS
        nb_threads = min(max_block_size, d_input.size)
        nb_blocks = (d_input.size + nb_threads - 1) // nb_threads

        extras = cuda.device_array(
            shape=nb_blocks, dtype=d_input.dtype
        )

        # Perform the scan on the GPU.
        start_event = cuda.event(True)
        start_event.record(stream=stream)

        kernel_step1[nb_blocks, nb_threads, stream](
            d_input, d_output, extras
        )

        if nb_blocks > 1:
            # Recursively scan the block totals.
            self(extras, extras, stream)

            # Apply the scanned totals of previous blocks.
            kernel_step2[nb_blocks, nb_threads, stream](
                d_output, extras
            )

        stop_event = cuda.event(True)
        stop_event.record(stream=stream)
        stop_event.synchronize()

        core.config.CUDA_LOW_OCCUPANCY_WARNINGS = _sav

        return cuda.event_elapsed_time(start_event, stop_event)

#### Tests — exercise 4 — the scan of any number of blocks

In [22]:
def test4(h_input, scan_class=InclusiveScan):
    """This one writes into the output array and returns only the timing."""
    def check4(result):
        verify(f"inclusive scan of {h_input.size} values", result,
               np.cumsum(h_input))

    scanner = scan_class(lambda a, b: a + b)
    d_output = cuda.device_array(shape=h_input.size, dtype=h_input.dtype)
    ct = scanner(cuda.to_device(h_input), d_output)
    check4(d_output.copy_to_host())
    return ct


rng = np.random.default_rng(4)
# Sizes that cross a block boundary, and one that does not divide it: the
# second pass over the block totals is where a full scan usually goes wrong.
for size in (256, 257, 1 << 12, (1 << 16) + 13):
    test4(rng.integers(0, 1 << 10, size=size, dtype=np.int32))

PASS  inclusive scan of 256 values  (256 values)
PASS  inclusive scan of 257 values  (257 values)
PASS  inclusive scan of 4096 values  (4096 values)
PASS  inclusive scan of 65549 values  (65549 values)


#### Benchmark — the scan of any number of blocks

In [23]:
def build(size):
    h_input = np.random.default_rng(size).integers(
        0, 1 << 10, size=size, dtype=np.int32)
    d_input = cuda.to_device(h_input)
    d_output = cuda.device_array(shape=size, dtype=np.int32)
    scanner = InclusiveScan(lambda a, b: a + b)
    # A scan reads the input and writes an output of the same size, so the
    # traffic is twice the array -- plus the second pass over the block totals.
    return ((lambda: scanner(d_input, d_output)),
            (lambda: np.cumsum(h_input)), 2 * h_input.nbytes)


benchmark("inclusive scan: GPU against numpy.cumsum",
          build, (1 << 20, 1 << 22, 1 << 24, 1 << 26))

inclusive scan: GPU against numpy.cumsum   —   this card peaks at 448 GB/s
        size |  GPU (ms) |  NumPy (ms) |  speed-up |    GB/s |  of peak
------------------------------------------------------------------------
     1048576 |     0.459 |       3.961 |      8.6x |    18.3 |     4.1%
     4194304 |     0.642 |      12.985 |     20.2x |    52.3 |    11.7%
    16777216 |    15.768 |      51.785 |      3.3x |     8.5 |     1.9%
    67108864 |     3.361 |     200.149 |     59.5x |   159.7 |    35.7%
